# F1 pit-stop strategy model

This notebook reads historical laps from the project SQLite database and reproduces the preserved Random Forest training and temporal evaluation. The pre-race application retrains as-of the selected event cutoff to prevent the evaluation race from entering its own training data.

In [ ]:
from services.database import initialize_schema, laps_frame
from data_preparation import build_pit_training_data
from train_models import train_lap_time_model, train_pit_stop_model

initialize_schema()
laps = laps_frame()
laps.shape

In [ ]:
coverage = (
    laps.groupby('Season')
        .agg(events=('Race', 'nunique'), drivers=('Driver', 'nunique'), lap_rows=('LapNumber', 'size'))
        .sort_index()
)
coverage

In [ ]:
pit_rows = build_pit_training_data(laps)
lap_model, lap_metrics = train_lap_time_model(laps)
pit_model, pit_metrics = train_pit_stop_model(pit_rows)
{'lap_time': lap_metrics, 'next_tyre': pit_metrics}

## Interpretation

The lap-time model is scored with a temporal season holdout when enough seasons exist. The next-tyre classifier uses held-out race transitions. MAE and accuracy describe historical holdout performance, not certainty for a future race. The strategy engine combines these estimates with circuit-specific tyre pace, degradation, inventory limits and pit loss.